# walter

LLM-assisted dataset construction for LASA drugs. *"Walter, are these two LASA drugs? Make no mistakes."*

All parameters, paths, and column names are in **`config.py`** — that is the single source of truth. Edit there, not here.

---

## Nomenclature and Terminology

**Raw registries** $\mathcal{R}^{\text{PH}}_{\text{raw}}$ and $\mathcal{R}^{\text{US}}_{\text{raw}}$ refer to the one-column drug name CSVs for the Philippine FDA and US FDA sources, stored at `_data/R_ph.csv` and `_data/R_us.csv` respectively. Only one is active per run, selected via `DATA_SOURCE` in `config.py`.

**$\mathcal{R}_{\text{clean}}$** is the result of passing the active raw registry through the preprocessing pipeline: lowercased, symbols stripped, duplicates removed. It is an in-memory intermediate only and is not saved to disk. It serves as the sampling pool from which $U$ is drawn.

The final dataset $D$ has columns `x_1, t_1, x_2, t_2, label`, where `x_1` and `x_2` are drug names, `t_1` and `t_2` are their IPA transcriptions, and `label` is the pair's class. $D$ is the union of:

- **$P$** — confirmed LASA pairs. Registry-independent: sourced either from a pre-existing file (`_data/P.csv`, columns `x_1, x_2`) or proposed by a local LLM using $\mathcal{R}_{\text{clean}}$ as its candidate pool. All $P$ rows carry `label = 1`.
- **$U$** — similarity-filtered unlabeled pairs drawn from $\mathcal{R}_{\text{clean}}$ via two-tier sampling. $U$ may contain undetected true LASA pairs. All $U$ rows carry `label = 0` (unlabeled, **not** confirmed negative).

$|U| \gg |P|$, $\quad P \cap U = \emptyset$.

Both $P$ and $U$ are recoverable from $D$ by filtering on `label`. Only $D$ is saved to disk (`_results/D.csv`).

## 1. Project Setup

In [1]:
%load_ext autoreload
%autoreload 2

## 2. Configuration

All settings are in `config.py`. The cell below just prints the active config so you can confirm before running.

In [2]:
from config import (
    DATA_SOURCE,
    FROM_FILE,
    UNLABELED_TO_POSITIVE_RATIO,
    TIER_2_SAMPLE_SIZE,
    SEED,
    P_INPUT_CSV,
    D_OUT_CSV,
)

print(f"Data source    : {DATA_SOURCE.name}")
print(
    f"P from file    : {FROM_FILE}  {'→ ' + str(P_INPUT_CSV) if FROM_FILE else '(LLM proposer)'}"
)
print(f"U/P ratio      : 1:{UNLABELED_TO_POSITIVE_RATIO}")
print(f"Tier 2 sample  : {TIER_2_SAMPLE_SIZE:,}")
print(f"Seed           : {SEED}")
print()
print(f"Output: D → {D_OUT_CSV}")

Data source    : PH
P from file    : True  → _data\P.csv
U/P ratio      : 1:30
Tier 2 sample  : 10,000
Seed           : 42

Output: D → _results\D.csv


## 3. Preprocessing

Loads the active raw registry (`_data/R_{ph|us}.csv`), normalizes and cleans all drug names
into $\mathcal{R}_{\text{clean}}$. Not saved — in-memory only.

In [3]:
import src.preprocessing as pre

R_clean = pre.run(source=DATA_SOURCE)
print(f"\nCleaned registry: {len(R_clean):,} drug names")
display(R_clean.head(10))

[preprocessing] Source: PH
[preprocessing] Rows: 22,837 raw → 22,455 clean  (dropped 382)

Cleaned registry: 22,455 drug names


,drug_name
0,09 sodchlorsaph
1,1 ceeplus
2,1000vc
3,2 gen
4,2 gen 750
5,2 gen scp
6,2 king
7,2 max
8,24 alkaline c
9,2baconil tts10


## 4. Confirmed LASA Pairs (P)

- `FROM_FILE = True` → reads `_data/P.csv` (must have columns `x_1`, `x_2`)
- `FROM_FILE = False` → generates pairs via the local LLM proposer

Set `FROM_FILE` in `config.py`.

In [4]:
import pandas as pd
from config import FROM_FILE

if FROM_FILE:
    from config import P_INPUT_CSV

    if not P_INPUT_CSV.exists():
        raise FileNotFoundError(
            f"{P_INPUT_CSV} not found. "
            "Place your confirmed LASA pairs CSV there, "
            "or set FROM_FILE = False in config.py to use the LLM proposer."
        )
    P = pd.read_csv(P_INPUT_CSV)
    print(f"Loaded P from {P_INPUT_CSV}: {len(P):,} pairs")
else:
    from config import LLM_OUTPUT_JSON
    from src.proposer.llm import LocalModel
    from src.proposer.inference import run_inference, load_inference

    run_inference(
        registry_df=R_clean,
        model_choice=LocalModel.QWEN3_1_7B,
    )
    P = load_inference(LLM_OUTPUT_JSON)
    print(f"Generated P via LLM: {len(P):,} pairs")

display(P.head())
print("Columns:", list(P.columns))

Loaded P from _data\P.csv: 536 pairs


,x_1,x_2
0,abelcet,amphotericin b
1,accupril,aciphex
2,acetaminophen,acetazolamide
3,acetazolamide,acetohexamide
4,acetic acid for irrigation,glacial acetic acid


Columns: ['x_1', 'x_2']


## 5. Unlabeled Pairs (U)

Constructs U via two-tier similarity-filtered sampling.
Parameters come from `config.py` (`UNLABELED_TO_POSITIVE_RATIO`, `TIER_2_SAMPLE_SIZE`, etc.).

In [5]:
import src.noise as noise
from config import UNLABELED_TO_POSITIVE_RATIO, TIER_2_SAMPLE_SIZE, SEED

U = noise.make_noise(
    pairs_df=P,
    registry_df=R_clean,
    ratio=UNLABELED_TO_POSITIVE_RATIO,
    tier_2_sample_size=TIER_2_SAMPLE_SIZE,
    seed=SEED,
)

display(U.head())


[noise] P-vocabulary size   : 784
[noise] Known positive pairs: 534
[noise] Registry size       : 22,455
[noise] Outside vocab       : 22,336
[noise] Target |U|          : 16,020  (ratio 1:30)
[noise] Similarity threshold: 20 (ANY measure)
[noise] Tier 2 sample size  : 10,000

[noise] Building Tier 1 (anchor-based hard negatives)...
[noise] Tier 1 candidates: 14,037,458
[noise] Building Tier 2 (broader coverage, sample=10,000)...
[noise] Tier 2 candidates: 39,041,662

[noise] Final Tier 1: 10,413
[noise] Final Tier 2: 5,607
[noise] Total U     : 16,020  (actual ratio 1:30.0)


,x_1,x_2,similarity,tier,label
0,rimantadine,mtrex,37.500000,1,0
1,carboplatin,citrupro cap,34.782609,1,0
2,allegra anti-itch cream,xensal plus,38.571429,1,0
3,tizanidine,jensar,25.714286,1,0
4,floranex,metromic forte,55.384615,1,0


## 6. Assemble and Save

Cleans, deduplicates, adds IPA transcriptions, and saves `_results/D.csv`.
P and U are recoverable from D by filtering on `label` (1 and 0 respectively).

In [6]:
from src.dataset import assemble_and_save

D = assemble_and_save(P, U, add_phonemes=True)

display(D.head(10))
print(f"\nD shape: {D.shape}")
print(D["label"].value_counts().to_string())

[dataset] Removed 11 self-pairs

[dataset] P (clean): 525 pairs
[dataset] U (clean): 16,020 pairs
[dataset] D (union, deduped): 16,545 pairs
[dataset]   label=1 (P): 525
[dataset]   label=0 (U): 16,020

[dataset] Adding IPA transcriptions...
[phonemes] Unique drug names: 13,221
[phonemes] Batch size       : 256
     256 / 13,221  (1.9%)  [0.8s]
     512 / 13,221  (3.9%)  [1.2s]
     768 / 13,221  (5.8%)  [1.5s]
   1,024 / 13,221  (7.7%)  [1.9s]
   1,280 / 13,221  (9.7%)  [2.3s]
   1,536 / 13,221  (11.6%)  [2.8s]
   1,792 / 13,221  (13.6%)  [3.1s]
   2,048 / 13,221  (15.5%)  [3.4s]
   2,304 / 13,221  (17.4%)  [3.7s]
   2,560 / 13,221  (19.4%)  [4.0s]
   2,816 / 13,221  (21.3%)  [4.3s]
   3,072 / 13,221  (23.2%)  [4.5s]
   3,328 / 13,221  (25.2%)  [5.1s]
   3,584 / 13,221  (27.1%)  [5.4s]
   3,840 / 13,221  (29.0%)  [5.6s]
   4,096 / 13,221  (31.0%)  [5.9s]
   4,352 / 13,221  (32.9%)  [6.2s]
   4,608 / 13,221  (34.9%)  [6.6s]
   4,864 / 13,221  (36.8%)  [7.3s]
   5,120 / 13,221  (38.7%) 

,x_1,t_1,x_2,t_2,label
0,feverfree,fiːvɚfɹiː,tramexol,tɹeɪmksɑːl,0
1,sinequan,saɪŋkwən,vasatraz,væsɐtɹæz,0
2,fentanyl,fɛntɐnaɪl,amlothix,æmloʊðɪks,0
3,coxto,kɑːkstoʊ,per c plus,pɜː siː plʌs,0
4,paradrin,pæɹədɹɪn,vercin,vɜːsɪn,0
5,tetanus diptheria toxoid,tɛtənəs dɪpθɜːɹiə tɑːksɔɪd,myzin,maɪzɪn,0
6,engerix b pediatric adolescent,ɛndʒɛɹɪks biː piːdɪætɹɪk ædəlɛsənt,fenodix,fɛnədɪks,0
7,leucovorin calcium,luːkəvɔːɹɪn kælsiəm,garda,gɑːɹdə,0
8,clotriva 6,klɑːtɹɪvə sɪks,rosumax 20,ɹɑːsuːmæks twɛnti,0
9,stabigran 110,stæbɪgɹən wʌnhʌndɹɪd tɛn,letero,lɛtɛɹoʊ,0



D shape: (16545, 5)
label
0    16020
1      525
